# Get Candidate Synonyms (v4 - Enhanced Boundaries)

**Purpose**: Generate candidate synonym phrases for each category using Gemini API.

**Updates in v4:**
- Enhanced prompt with explicit inclusion/exclusion criteria
- Boundary-aware generation to minimize cross-category overlaps  
- American English spelling standardization
- Emphasis on category-specific uniqueness
- Output: `11_candidate_phrases_v4.csv` (200 phrases per category)

**Changes from v3:**
- Added include_criteria and exclude_criteria to prompt
- Instructions to avoid overlapping with other categories
- Focus on PRIMARY cause identification
- Stricter specificity requirements

**Process:**
1. Load taxonomy with definitions and seed keywords
2. For each category, call Gemini API to generate 200 candidate phrases
3. Save results for similarity analysis in notebook 12

In [13]:
import google.generativeai as genai
import pandas as pd
from dotenv import load_dotenv
import os
import json

In [14]:
# Configuration for Google Generative AI
# Load environment variables
load_dotenv(dotenv_path='.env.local')
api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=api_key)
print("Google Generative AI configured.")

Google Generative AI configured.


In [15]:
# API call function to get candidate synonyms 
def generate_synonyms(category_name: str, definition: str, include_criteria: str, exclude_criteria: str, seed_keywords: list) -> list:
    """
    Calls the Gemini API with a rich context prompt to generate candidate synonyms.

    Args:
        category_name: The name of the reputational risk category.
        definition: The definition of the category.
        include_criteria: What should be included in this category.
        exclude_criteria: What should be excluded from this category.
        seed_keywords: A list of example seed keywords for context.

    Returns:
        A list of generated keyword phrases.
    """
    model = genai.GenerativeModel(MODEL_NAME)

    # Format the seed keywords for clear presentation in the prompt
    seed_keyword_str = "\n- ".join(seed_keywords)

    # Create improved prompt with category boundaries
    prompt = f"""You are an expert corporate risk analyst creating a keyword dictionary for text classification. 

**YOUR TASK**: Generate exactly {TARGET_PHRASE_COUNT} highly specific phrases for the "{category_name}" category.

**CATEGORY DEFINITION**: 
Events where the root causes are {definition}

**INCLUSION CRITERIA** (What belongs in THIS category):
{include_criteria}

**EXCLUSION CRITERIA** (What does NOT belong - these go in other categories):
{exclude_criteria}

**EXAMPLE SEED KEYWORDS**:
- {seed_keyword_str}

**OTHER CATEGORIES TO AVOID OVERLAPPING WITH**:
Governance, Personnel, Products, IT/Data, Processes, Legal, Communication (whichever are NOT {category_name})

**CRITICAL REQUIREMENTS**:
1. Generate ONLY phrases that fit the {category_name} category based on the inclusion criteria
2. DO NOT generate phrases that fit the exclusion criteria or other categories
3. Be SPECIFIC - avoid generic phrases that could apply to multiple categories
4. Use American English spelling (e.g., "unauthorized" not "unauthorised")
5. Vary the wording - don't just add prefixes/suffixes to seed keywords
6. Focus on phrases that clearly indicate {category_name} as the PRIMARY cause of the reputational issue
7. Each phrase should be 2-7 words long
8. Do NOT generate exact duplicates of seed keywords

**OUTPUT FORMAT**:
Return ONLY a valid JSON list of exactly {TARGET_PHRASE_COUNT} unique strings. No explanations, no markdown formatting.

Example format: ["phrase 1", "phrase 2", "phrase 3", ...]"""

    try:
        response = model.generate_content(prompt)
        cleaned_response = response.text.strip().replace('```json', '').replace('```', '').strip()
        generated_phrases = json.loads(cleaned_response)
        return generated_phrases if isinstance(generated_phrases, list) else []
    except Exception as e:
        print(f"An error occurred for category '{category_name}': {e}")
        return []


In [16]:
# Function to build a dictionary from taxonomy file 
def expand_list_from_taxonomy(file_path: str):
    """
    Loads a CSV, iterates through categories, and calls the Gemini API to build a balanced dictionary.
    """
    try:
        df = pd.read_csv(file_path)
        print(f"\nSuccessfully loaded '{file_path}'.")
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return None

    ai_keyword_list = {}
    print("--- Starting keyword generation with enhanced boundary-aware prompts ---")

    # Iterate through each row of the DataFrame
    for index, row in df.iterrows():
        cat_name = row['category']
        cat_def = row['definition']
        cat_include = row['include_criteria']
        cat_exclude = row['exclude_criteria']
        # Split the newline-separated keywords string into a clean list
        cat_seeds = [keyword.strip() for keyword in row['keywords'].split('\n') if keyword.strip()]

        print(f"\nProcessing Category: '{cat_name}'...")
        phrases = generate_synonyms(cat_name, cat_def, cat_include, cat_exclude, cat_seeds)

        if phrases:
            ai_keyword_list[cat_name] = phrases
            print(f"  > Successfully generated {len(phrases)} phrases.")
        else:
            print(f"  > Failed to generate phrases for '{cat_name}'.")

    return ai_keyword_list


In [17]:
# # Define variables
TAXONOMY_FILE = 'data/00_raw/taxonomy_sheet_v2.csv'  # Updated to new 7-category taxonomy
MODEL_NAME = 'gemini-2.5-flash' # latest model as of June 2025, faster than the Pro version and good for most tasks
TARGET_PHRASE_COUNT = 200 # Aim for 200 candidate phrases per category
CANDIDATE_PHRASE_FILE = 'results/phase i/11_candidate_phrases_v4.csv'  # v4: Enhanced with boundary-aware prompts

# Run the function to build the list
candidate_synonyms = expand_list_from_taxonomy(TAXONOMY_FILE)

if candidate_synonyms:
    print("\n\n--- AI-Generated Balanced Dictionary (Final Output) ---")
    for category, phrases in candidate_synonyms.items():
        print(f"\n--- Category: {category} ---")
        print(f"    Number of phrases: {len(phrases)}")
    # Save the candidate phrases to a CSV file
    df = pd.DataFrame.from_dict(candidate_synonyms, orient='index').transpose()
    df.to_csv(CANDIDATE_PHRASE_FILE, index=False)
    print(f"\nCandidate phrases saved to '{CANDIDATE_PHRASE_FILE}'.")


Successfully loaded 'data/00_raw/taxonomy_sheet_v2.csv'.
--- Starting keyword generation with enhanced boundary-aware prompts ---

Processing Category: 'Governance'...
  > Successfully generated 200 phrases.

Processing Category: 'Personnel'...
  > Successfully generated 200 phrases.

Processing Category: 'Products'...
  > Successfully generated 200 phrases.

Processing Category: 'IT/Data'...
  > Successfully generated 200 phrases.

Processing Category: 'Processes'...
  > Successfully generated 200 phrases.

Processing Category: 'Legal'...
  > Successfully generated 200 phrases.

Processing Category: 'Communication'...
  > Successfully generated 200 phrases.


--- AI-Generated Balanced Dictionary (Final Output) ---

--- Category: Governance ---
    Number of phrases: 200

--- Category: Personnel ---
    Number of phrases: 200

--- Category: Products ---
    Number of phrases: 200

--- Category: IT/Data ---
    Number of phrases: 200

--- Category: Processes ---
    Number of phrases: 

In [18]:
# Convert candidate synonyms to a DataFrame with 2 columns: 'category' and 'candidate_phrase'
df = pd.DataFrame([(cat, phrase) for cat, phrases in candidate_synonyms.items() for phrase in phrases], 
                  columns=['category', 'candidate_phrase'])
df.head()

,category,candidate_phrase
0,Governance,Board oversight failure
1,Governance,Weak corporate governance
2,Governance,Inadequate board supervision
3,Governance,Non-compliant governance practices
4,Governance,Poor risk culture oversight


In [19]:
# Save the DataFrame to a CSV file
df.to_csv(CANDIDATE_PHRASE_FILE, index=False)

In [20]:
# Display final statistics
print("=" * 80)
print("CANDIDATE PHRASE GENERATION COMPLETE (v4)")
print("=" * 80)
print(f"\n📊 Statistics:")
print(f"  Categories: {len(df['category'].unique())}")
print(f"  Total phrases generated: {len(df):,}")
print(f"\nPhrases per category:")
print(df['category'].value_counts().sort_index())
print(f"\n✅ Output saved to: {CANDIDATE_PHRASE_FILE}")
print(f"\n💡 Next step: Run 11b_validate_candidate_phrases.ipynb to check for overlaps")
print(f"\n✅ Output saved to: {CANDIDATE_PHRASE_FILE}")
print(f"\n🎯 Next step: Run notebook 12 to calculate semantic similarities")

CANDIDATE PHRASE GENERATION COMPLETE (v4)

📊 Statistics:
  Categories: 7
  Total phrases generated: 1,400

Phrases per category:
category
Communication    200
Governance       200
IT/Data          200
Legal            200
Personnel        200
Processes        200
Products         200
Name: count, dtype: int64

✅ Output saved to: results/phase i/11_candidate_phrases_v4.csv

💡 Next step: Run 11b_validate_candidate_phrases.ipynb to check for overlaps

✅ Output saved to: results/phase i/11_candidate_phrases_v4.csv

🎯 Next step: Run notebook 12 to calculate semantic similarities


## Summary

**Output file**: `results/phase i/11_candidate_phrases_v3.csv`

**Statistics:**
- Target phrases per category: 200
- Total categories: 7
- Expected total phrases: ~1,400

**7 Categories:**
1. Governance
2. Personnel
3. Products
4. IT/Data
5. Processes
6. Legal
7. Communication

**Next step**: Run notebook 12 to calculate semantic similarities between candidates and seed keywords.